# Publish the clean-split DistilBERT to the Hugging Face Hub

Runs on a Colab **CPU** runtime (no GPU, no training), in about 15–30 minutes, mostly waiting for
the demo to rebuild. It runs `scripts/publish_model.py`, which stops at the first failed check.

1. **Check, no writes.**
   - The run folder in Drive is the scored run: its records match the committed bundle.
   - An agreement check on 1,024 examples: the weights reproduce the saved test logits
     (absolute tolerance 1e-3, every prediction equal).
   - The Hub still holds the legacy model.
   - The demo answers, its Gradio version is read, and its patch applies.
2. **Upload.**
   - The legacy commit of `krimits/distilbert-hotel-reviews` gets the tag `legacy-split-v1`.
   - One commit holds the new model and its card. It is checked on the commit it returns:
     files, card, and the SHA-256 of the weights downloaded from it.
3. **Pin the demo.** `krimits/hotel-review-demo` loads the model at that exact commit, and its texts
   say which model it is.
4. **Check the demo.** It must report that commit and those weights, then label one positive and
   one negative review.

**Interrupted?** Run the notebook again from the top. `publish_state.json` in Drive records every
completed write. Each step checks the Hub again, and is skipped only if it was done exactly as
expected.

**Token.** It needs a Hugging Face token with write access: a Colab secret named `HF_TOKEN`, or
paste one when asked.

In [1]:
RUN_DIR = "/content/drive/MyDrive/hotel-review-nlp-phase2/distilbert_v2"  # the random split, full fine-tune
STATE_PATH = "/content/drive/MyDrive/hotel-review-nlp-phase2/publish_state.json"
DRY_RUN = False  # True: only the checks and the card; nothing is written to the Hub or the demo

In [2]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
assert Path(RUN_DIR, "model.safetensors").is_file(), f"no weights in {RUN_DIR}"

Mounted at /content/drive


In [3]:
import os
import shlex
import subprocess
import sys

REPO = Path("/content/hotel-review-nlp")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", "--branch", "claude/funny-rubin-31xgtb",
                    "https://github.com/krimits/hotel-review-nlp.git", str(REPO)], check=True)
os.chdir(REPO)
# The combination the tests ran with.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub==0.36.2",
                "transformers==4.56.2", "gradio_client==1.13.3"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "."], check=True)
print("publishing code:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

publishing code: 21d535ceafb6d1b95d4747d806c6153c2471718d


In [4]:
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:  # no secret, or no access to it
    token = None
if token:
    os.environ["HF_TOKEN"] = token
else:
    from huggingface_hub import notebook_login

    notebook_login()

In [5]:
from huggingface_hub import HfApi

print("Hugging Face account:", HfApi().whoami()["name"])

Hugging Face account: krimits


## The test set, for the agreement check

In [6]:
def run(command: str) -> None:
    """Run a command and show its output as it comes."""
    print("$", command, flush=True)
    with subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode:
        raise RuntimeError(f"stopped with exit code {process.returncode}: {command}")


run("python scripts/fetch_booking_515k.py")
if not Path("data/processed_v2/test.parquet").exists():
    run("python -m reviewnlp.data.preprocess --config configs/baselines.yaml")

$ python scripts/fetch_booking_515k.py
/content/hotel-review-nlp/data/raw/booking_reviews_515k.csv: 238,154,765 bytes, SHA-256 a4810c2757934f0a826a1b16a437eb67a38be45b1a22ad56772afce0b6c11af9 (Dricz/515k-Hotel-Reviews-In-Europe at cdaea79)
$ python -m reviewnlp.data.preprocess --config configs/baselines.yaml
<frozen runpy>:130: RuntimeWarning: 'reviewnlp.data.preprocess' found in sys.modules after import of package 'reviewnlp.data', but prior to execution of 'reviewnlp.data.preprocess'; this may result in unpredictable behaviour
Loading data/raw/booking_reviews_515k.csv ...
Labeled (unambiguous) reviews: 145,877
{
  "train": {
    "rows": 117880,
    "positive": 91775,
    "negative": 26105,
    "avg_chars": 119.85122158126909
  },
  "dev": {
    "rows": 14734,
    "positive": 11471,
    "negative": 3263,
    "avg_chars": 119.74616533188544
  },
  "test": {
    "rows": 13263,
    "positive": 10000,
    "negative": 3263,
    "avg_chars": 121.4918947447787
  }
}


## 1. Check (no writes)

In [7]:
PUBLISH = f"python -u scripts/publish_model.py {{step}} --run-dir {shlex.quote(RUN_DIR)} --state {shlex.quote(STATE_PATH)}"
run(PUBLISH.format(step="check") + " --card-out /content/model_card.md")
print(Path("/content/model_card.md").read_text(encoding="utf-8"))

$ python -u scripts/publish_model.py check --run-dir /content/drive/MyDrive/hotel-review-nlp-phase2/distilbert_v2 --state /content/drive/MyDrive/hotel-review-nlp-phase2/publish_state.json --card-out /content/model_card.md
2026-09-29 09:02:43.586442: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
check: agreement check on 1,024 examples: largest difference 3.635883331298828e-06, 1024 of 1024 predictions equal
check: the legacy model is on main
check: demo unpatched at f4362f1351cb85427aae352cdb36666294172d44, Gradio 6.27.0
---
license: mit
base_model: distilbert-base-uncased
language:
  - en
tags:
  - sentiment-analysis
  - text-classification
  - hotel-reviews
library_name: transformers
pipeline_tag: text-classification
---

# DistilBERT hotel-r

## 2–4. Upload, pin the demo, check the demo

In [8]:
if DRY_RUN:
    print("Dry run: nothing was written to the Hub or the demo.")
else:
    run(PUBLISH.format(step="upload"))
    run(PUBLISH.format(step="pin-demo"))
    run(PUBLISH.format(step="check-demo"))

$ python -u scripts/publish_model.py upload --run-dir /content/drive/MyDrive/hotel-review-nlp-phase2/distilbert_v2 --state /content/drive/MyDrive/hotel-review-nlp-phase2/publish_state.json
2026-09-29 09:06:01.547723: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
check: agreement check on 1,024 examples: largest difference 3.635883331298828e-06, 1024 of 1024 predictions equal
check: the legacy model is on main
check: demo unpatched at f4362f1351cb85427aae352cdb36666294172d44, Gradio 6.27.0

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            


  ...bert_v2/model.safetensors:   0%|          |  574kB /  268MB            


  ...bert_v2/model.safetensors:

In [9]:
import json

state = json.loads(Path(STATE_PATH).read_text(encoding="utf-8"))
keys = ("code_commit", "old_hub_commit", "old_model_sha256", "model_commit", "new_model_sha256",
        "legacy_tag", "space_parent_commit", "space_commit", "gradio_version", "logits_check", "demo_check")
print(json.dumps({key: state.get(key) for key in keys}, indent=1))

{
 "code_commit": "21d535ceafb6d1b95d4747d806c6153c2471718d",
 "old_hub_commit": "9fe2f7f9963c206294200880b2401347804a80d6",
 "old_model_sha256": "f30daa44e795b7015151892ff5ec285e14a916e122902aa0617a15bf96ec54b7",
 "model_commit": "7306aebcaaebc00d579f5d0a91001ae376f18158",
 "new_model_sha256": "f2a2bbc9a31c8f3b1d9bee46ec11bb3d8942e7e59c0ef4711c3cb2a6c1cf81c2",
 "legacy_tag": {
  "commit": "9fe2f7f9963c206294200880b2401347804a80d6",
  "name": "legacy-split-v1"
 },
 "space_parent_commit": "f4362f1351cb85427aae352cdb36666294172d44",
 "space_commit": "328a3065eb69b34914276330e25f420c7bdede33",
 "gradio_version": "6.27.0",
 "logits_check": {
  "argmax_agree": 1024,
  "atol": 0.001,
  "examples": 1024,
  "finite": true,
  "max_abs_diff": 3.635883331298828e-06,
  "name": "agreement check on 1,024 examples",
  "passed": true,
  "rtol": 0.0,
  "same_shape": true,
  "within_tolerance": true
 },
 "demo_check": {
  "checked_at": "2026-09-29T09:13:27+00:00",
  "model_info": {
   "distilbert_repo":